# Lab 25 · Use your gateway's models and tools in Claude Code

Publish Claude Opus 5.5, Sonnet 5.5, and Haiku 5.5 from Microsoft Foundry through your AI Gateway, run Claude Code on
them, and give it the Microsoft Learn tools from an MCP server on the same gateway. Then push the same key into the
gateway's token limit and budget, see what the gateway answers and how Claude Code reports it, and cut Claude Code off
by suspending its key. This notebook is the code track of the [lab guide](README.md): it deploys
[infra/claude-code.bicep](infra/claude-code.bicep), calls the gateway's Anthropic route with the Anthropic SDK, and
runs Claude Code in non-interactive mode with `claude -p`.

**Publish the Claude models → Run Claude Code on them → Add the Learn tools → Burst Haiku until 429 → Spend the Opus budget until 403 → Suspend the key**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| [infra/claude-code.bicep](infra/claude-code.bicep) | Grants the gateway's managed identity the Foundry User role on the account, creates a provider named after the account, publishes the alias `<account>/<deployment>` for each Claude deployment with a token limit of 100,000 tokens per minute for each caller, adds a budget of $5 per day for each caller to Opus, and creates the runtime key `claude-code`. |
| Model endpoint `<gateway-url>/default/models/anthropic` | The gateway's Anthropic route, which serves the Messages API. Claude Code and the Anthropic SDK add `/v1/messages`, and send the key as a bearer token. |
| MCP endpoint `<gateway-url>/default/toolservers/learn/mcp` | The `learn` MCP server from lab 04, in front of the public Microsoft Learn MCP server. Claude Code sends the same key as a bearer token. |
| Claude Code | Runs with `claude -p` and five `ANTHROPIC_*` settings, so its agent loop runs on your machine and its model calls go to the gateway. |
| Anthropic SDK | Sends the bursts that run into the limits, with the same key, so you can read each status and header the gateway answers. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, Bicep deployments, keys, and retries. |

![Apps, agents, and coding agents such as Claude Code call Azure AI Gateway with a key or a Microsoft Entra ID token. The gateway applies access, aliases, MCP federation, token and cost limits, and observability, then calls the claude-opus-5-5, claude-sonnet-5-5, and claude-haiku-5-5 deployments in Microsoft Foundry with its managed identity, and the Microsoft Learn MCP server](../assets/diagrams/claude-code.svg)

## Prerequisites

- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`, with
  Contributor on the gateway's resource group, and Owner or User Access Administrator on the Foundry account, so
  the template can grant the gateway's managed identity the Foundry User role. If the role is already assigned, for
  example by the portal's import, the notebook skips it.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) has run, so the gateway exists.
- A Foundry account with three Claude deployments, each named after its model: `claude-opus-5-5`,
  `claude-sonnet-5-5`, and `claude-haiku-5-5`. See
  [Deploy and use Claude models in Microsoft Foundry](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-claude).
  Unless they're in the labs' default account, set `AIGW_FOUNDRY_ACCOUNT` and `AIGW_FOUNDRY_RESOURCE_GROUP` to the
  account and its resource group before you start Jupyter. Model inference is billed per use.
- Optional: [Claude Code](https://code.claude.com/docs/en/setup), tested with 2.1.314, on the kernel's `PATH`.
  Without it, the notebook skips the Claude Code steps and still runs the SDK steps. You don't need an Anthropic
  account or a Claude subscription.

The notebook gives Claude Code its own temporary `CLAUDE_CONFIG_DIR`, so it doesn't read or change your Claude Code
settings, sign-in, sessions, or MCP servers. It also leaves your `ANTHROPIC_*` variables out of Claude Code's
environment: Claude Code would send `ANTHROPIC_API_KEY` in an `x-api-key` header, which the gateway rejects.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it reuses what exists.

In [ ]:
%pip install -q --upgrade anthropic azure-identity requests matplotlib

### 0. Initialize the lab

`utils.Lab` finds the gateway, and the Foundry account in `AIGW_FOUNDRY_ACCOUNT`. The template names the provider
after the account, and each alias after the account and the deployment, such as `<account>/claude-opus-5-5`: the
same names that the portal's import creates. Claude Code picks a model by tier, so `aliases` maps each tier to its
alias. If `pip` upgraded a package that the kernel had already loaded, restart the kernel before you run this cell.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
from collections import Counter
from pathlib import Path

import anthropic
import requests
from IPython.display import display

sys.path.insert(0, "../shared")   # helpers and brand styles that every lab notebook shares
import brand
import utils

lab = utils.Lab("25-claude-code")
account     = lab.foundry_account   # the Foundry account with the Claude deployments, and the provider's name
models      = {"opus": "claude-opus-5-5", "sonnet": "claude-sonnet-5-5", "haiku": "claude-haiku-5-5"}
aliases     = {tier: f"{account}/{model}" for tier, model in models.items()}   # what Claude Code sends as the model
key_name    = "claude-code"         # the runtime key that Claude Code uses
server_name = "learn"               # the Microsoft Learn MCP server from lab 04

### 1. Check the Claude deployments

The template reads each deployment to register it on the gateway, so the three deployments must exist in the account.
The cell lists them with their model, version, and SKU.

In [ ]:
lab.require(lab.foundry_id, "Set AIGW_FOUNDRY_ACCOUNT and AIGW_FOUNDRY_RESOURCE_GROUP to the account with the "
            "Claude deployments, then restart the kernel.", utils.COGNITIVE_API)
deployments = {d["name"]: d for d in lab.arm("GET", f"{lab.foundry_id}/deployments",
                                             api_version=utils.COGNITIVE_API)["value"]}
rows = []
for model in models.values():
    found = deployments.get(model)
    ready = found is not None and found["properties"].get("provisioningState") == "Succeeded"
    rows.append((brand.mono(model),
                 found["properties"]["model"]["name"] if found else "-",
                 found["properties"]["model"].get("version", "-") if found else "-",
                 found.get("sku", {}).get("name", "-") if found else "-",
                 brand.chip("Ready", "ok") if ready else brand.chip("Missing", "bad")))
display(brand.table(f"Claude deployments in {account}", ["Deployment", "Model", "Version", "SKU", "State"], rows,
                    sub=lab.foundry_resource_group))
missing = [model for model in models.values() if model not in deployments]
assert not missing, f"Deploy {', '.join(missing)} in {account} first, as the lab guide's Before you begin shows."

### 2. Publish the Claude models for Claude Code

[infra/claude-code.bicep](infra/claude-code.bicep) gives the gateway's managed identity the Foundry User role on the
account, so the gateway calls Foundry without keys. It creates the provider `<account>`, registers the three
deployments as Anthropic Messages models, and publishes an alias for each one with a token limit of 100,000 tokens
per minute for each caller. Opus, the default and the most expensive model, also gets a budget of $5 per day for each
caller, with the `id` `claude-code-budget`. The template also creates the key `claude-code`.

The cell checks whether the role is already assigned, and records what already existed, so **Clean up** deletes only
what this notebook created. The deployment takes about a minute.

In [ ]:
FOUNDRY_USER = "53ca6127-db72-4b80-b1b0-d745d6d5456d"   # the Foundry User role
gateway_principal = lab.gateway["identity"]["principalId"]


def foundry_user_assignments():
    # The Foundry User assignments of the gateway's managed identity that cover the account
    found = lab.arm("GET", f"{lab.foundry_id}/providers/Microsoft.Authorization/roleAssignments",
                    api_version=utils.ROLES_API, params={"$filter": f"principalId eq '{gateway_principal}'"})["value"]
    return [a["id"] for a in found if a["properties"]["roleDefinitionId"].endswith(FOUNDRY_USER)]


def alias_path(tier):
    return f"{lab.workspace_id}/aliases/{account}-{models[tier]}"


def alias_policies(tier):
    return lab.arm("GET", alias_path(tier))["properties"]["policies"]


grant_role       = not foundry_user_assignments()
created_provider = lab.arm("GET", f"{lab.workspace_id}/modelProviders/{account}") is None
created_models   = [t for t, m in models.items() if lab.arm("GET", f"{lab.workspace_id}/modelProviders/{account}/models/{m}") is None]
created_aliases  = [t for t in models if lab.arm("GET", alias_path(t)) is None]
created_key      = lab.arm("GET", f"{lab.gateway_id}/apiKeys/{key_name}") is None

outputs = lab.deploy("infra/claude-code.bicep", gatewayName=lab.gateway_name, foundryAccountId=lab.foundry_id,
                     grantFoundryUser=grant_role, apiKeyName=key_name)
created_role = foundry_user_assignments() if grant_role else []
base_url = outputs["anthropicBaseUrl"]
original = {tier: alias_policies(tier) for tier in models}   # Clean up puts these back if a step leaves them changed


def limits(policies):
    tokens = next((f"{p['count']:,} tokens per {p['period']}" for p in policies if p["type"] == "tokenLimit"), "None")
    budget = next((f"{brand.usd(p['amount'])} per {p['period']}" for p in policies if p["type"] == "costLimit"), "None")
    return tokens, budget


display(brand.note("Gateway's managed identity: Foundry User on the account " +
                   ("granted by the template" if grant_role else "was already assigned"), "ok"))
display(brand.table("Aliases for Claude Code", ["Tier", "Alias", "Token limit for each caller", "Budget for each caller"],
                    [(brand.chip(tier, "violet"), brand.mono(aliases[tier]), *limits(original[tier])) for tier in models],
                    sub=base_url, badge=brand.chip(f"Provider {outputs['providerName']}", "brand")))

### 3. Test the Anthropic route, and see what one Opus call costs

Read the key, then send a Messages API request with the Anthropic SDK, the same way Claude Code does: to the base URL,
with the alias as `model`, the key as a bearer token, and the `anthropic-version` header, which the SDK adds. A new
key, new aliases, and a new role assignment can take a few minutes to work, so the cell retries until the gateway
answers `200`.

The gateway estimates what each call to a budgeted alias costs, and returns it in headers that start with
`x-alias-cost-per-day-`: `…-consumed` is this call's cost, and `…-remaining` is what's left of the caller's budget
today. Blocked calls get `…-retry-after` instead, which step 12 shows.

In [ ]:
COST = re.compile(r"^x-(?:alias-)?cost-per-\w+-(consumed|remaining|retry-after)$")


def cost_headers(headers):
    # The cost headers of a response, keyed by consumed, remaining, and retry-after
    return {match[1]: value for name, value in headers.items() if (match := COST.match(name.lower()))}


def send(alias, prompt, max_tokens=300, client=None):
    # Send one Messages API request, and return the status, headers, body, answer, tokens used, and estimated cost
    try:
        raw = (client or gateway).messages.with_raw_response.create(
            model=alias, max_tokens=max_tokens, messages=[{"role": "user", "content": prompt}])
        message = raw.parse()
        result = {"status": raw.status_code, "headers": raw.headers, "body": raw.http_response.text,
                  "text": "".join(block.text for block in message.content if block.type == "text"),
                  "model": message.model, "usage": (message.usage.input_tokens, message.usage.output_tokens)}
    except anthropic.APIStatusError as error:
        result = {"status": error.status_code, "headers": error.response.headers, "body": error.response.text,
                  "text": "", "model": alias, "usage": None}
    consumed = cost_headers(result["headers"]).get("consumed")
    result["tokens"] = sum(result["usage"]) if result["usage"] else None
    result["cost"] = float(consumed) if result["status"] == 200 and consumed else None
    return result


api_key, _ = lab.api_key(key_name)
display(brand.key(key_name, api_key, created_key))
gateway = anthropic.Anthropic(base_url=base_url, auth_token=api_key, max_retries=0)   # the key as a bearer token

first = {}
def first_call():
    first.update(send(aliases["opus"], "In one sentence, what is a code review for?", 120))
    return first["status"]

assert utils.wait_for_status(first_call, 200) == 200, first["body"]
display(brand.reply(first["text"], "Messages API through the gateway", model=first["model"], tokens=first["usage"],
                    sub=f"POST {base_url}/v1/messages"))
display(brand.info("What one Opus call costs", [(brand.mono(name), value) for name, value in first["headers"].items()
                                                if COST.match(name.lower())], sub=aliases["opus"],
                   badge=brand.chip(brand.usd(first["cost"]), "brand")))

### 4. Create the practice project

Claude Code works on a small project with two deliberate problems: `add` subtracts, and `divide` doesn't guard
against zero. The cell writes `~/contoso-calc/calc.py`, the same project that the guide uses, unless it already
exists.

In [ ]:
project = Path.home() / "contoso-calc"
created_project = not project.exists()
if created_project:
    project.mkdir(parents=True)
    (project / "calc.py").write_text("""\"\"\"Tiny calculator used to try Claude Code with models from Azure AI Gateway.\"\"\"


def add(a: float, b: float) -> float:
    return a - b


def divide(a: float, b: float) -> float:
    return a / b


if __name__ == "__main__":
    print(add(2, 3), divide(10, 2))
""")
display(brand.note(f"{'Created' if created_project else 'Reusing'} the practice project", "ok", project))

### 5. Point Claude Code at the gateway

Five settings send Claude Code to the gateway: the Anthropic route as `ANTHROPIC_BASE_URL`, the key as
`ANTHROPIC_AUTH_TOKEN`, which Claude Code sends as a bearer token, and an alias for each of its three tiers. Claude
Code then runs without an Anthropic sign-in. `run_claude` runs one prompt with `claude -p` in the practice project,
and reads what happened from Claude Code's JSON events: the answer, the aliases that answered, Claude Code's cost
estimate, the tools it called, and the HTTP status of any error.

Ask a quick question. Claude Code uses the `opus` tier by default, and the alias in the answer proves that the call
went through the gateway. The cost is Claude Code's own estimate, so it doesn't match the gateway's meter exactly.

In [ ]:
claude = shutil.which("claude")
claude_home = Path(tempfile.mkdtemp(prefix="claude-config-"))   # keeps your own Claude Code settings out of the runs
claude_settings = {
    "ANTHROPIC_BASE_URL":             base_url,
    "ANTHROPIC_AUTH_TOKEN":           api_key,
    "ANTHROPIC_DEFAULT_OPUS_MODEL":   aliases["opus"],
    "ANTHROPIC_DEFAULT_SONNET_MODEL": aliases["sonnet"],
    "ANTHROPIC_DEFAULT_HAIKU_MODEL":  aliases["haiku"],
    "CLAUDE_CONFIG_DIR":              str(claude_home),
    "AI_GATEWAY_API_KEY":             api_key,   # the MCP server in step 8 reads the key from here
}


def claude_env(settings=None):
    # Your environment without its ANTHROPIC_* and CLAUDE_CODE_USE_* variables, plus the gateway's settings
    env = {name: value for name, value in os.environ.items()
           if not name.startswith(("ANTHROPIC_", "CLAUDE_CODE_USE_", "CLAUDE_CONFIG_DIR"))}
    return {**env, **(claude_settings if settings is None else settings)}


def run_claude(prompt, *flags, settings=None):
    # Run claude -p in the practice project, and return what happened, read from its JSON events
    status = brand.live(brand.note(f"Claude Code: {prompt[:100]}", "wait"))
    start = time.time()
    done = subprocess.run([claude, "-p", prompt, *flags, "--output-format", "stream-json", "--verbose"],
                          cwd=project, env=claude_env(settings), capture_output=True, text=True, timeout=900)
    events = [json.loads(line) for line in done.stdout.splitlines() if line.startswith("{")]
    result = next((event for event in reversed(events) if event["type"] == "result"), {})
    ok = done.returncode == 0 and result.get("is_error") is False
    answer = result.get("result") or done.stderr.strip()[-300:]
    error = None if ok else (result.get("api_error_status") or
                             next((int(m[1]) for m in [re.search(r"API Error: (\d{3})", answer)] if m), None))
    run = {
        "ok":      ok,
        "status":  200 if ok else error,
        "answer":  answer,
        "models":  list(result.get("modelUsage") or {}),
        "cost":    result.get("total_cost_usd"),
        "turns":   result.get("num_turns"),
        "tools":   [block["name"] for event in events if event["type"] == "assistant"
                    for block in event["message"]["content"] if block["type"] == "tool_use"],
        "seconds": round(time.time() - start),
    }
    status.update(brand.note(f"Claude Code finished in {run['seconds']} s" + ("" if ok else f" with HTTP {error}"),
                             "neutral" if ok else "warn"))
    return run


def show_run(run, title):
    # Show Claude Code's answer, the alias that answered, and the tools it called, with where each one ran
    assert run["ok"], run["answer"]
    badge = brand.chip(f"Claude Code estimate {brand.usd(run['cost'], 4)}", "brand") if run["cost"] else None
    display(brand.reply(run["answer"], title, model=", ".join(run["models"]) or None, badge=badge, markdown=True,
                        sub=f"claude -p · {run['turns']} turns through the gateway in {run['seconds']} s"))
    if run["tools"]:
        display(brand.table("Tools that Claude Code called", ["Tool", "Calls", "Runs"], [
            (brand.mono(name), count, brand.chip("AI Gateway", "brand") if name.startswith("mcp__ai-gateway-") else "Your machine")
            for name, count in Counter(run["tools"]).items()]))


if claude:
    version = subprocess.run([claude, "--version"], capture_output=True, text=True).stdout.strip()
    display(brand.info("Claude Code settings", [
        ("Claude Code", brand.mono(version)),
        *[(name, brand.masked(value) if name.endswith(("TOKEN", "KEY")) else brand.mono(value))
          for name, value in claude_settings.items()],
    ]))
    show_run(run_claude("Reply with the single word pong"), "Claude Code · quick question")
else:
    display(brand.note("Claude Code isn't on the kernel's PATH: the notebook skips the Claude Code steps. "
                       "Install it, restart the kernel, and rerun to try them.", "warn"))

### 6. Run an agent task

Ask Claude Code to fix the project. `claude -p` can't ask for approval, so two flags give it permissions up front:
`--permission-mode acceptEdits` lets it edit files, and `--allowedTools "Bash(python3 *)"` lets it run Python. Claude
Code reads, plans, edits, and runs commands on your machine, and each of its model calls goes through the gateway.
Then the cell checks the agent's work itself.

In [ ]:
if claude:
    task = "Fix the bug in calc.py and add a guard for division by zero, then run python3 calc.py"
    show_run(run_claude(task, "--permission-mode", "acceptEdits", "--allowedTools", "Bash(python3 *)"),
             "Claude Code · agent task")

    ran = subprocess.run([sys.executable, "calc.py"], cwd=project, capture_output=True, text=True)
    probe = subprocess.run([sys.executable, "-c", "import calc\ntry:\n    print(repr(calc.divide(1, 0)))\n"
                            "except Exception as error:\n    print(type(error).__name__, error)"],
                           cwd=project, capture_output=True, text=True).stdout.strip()
    work = [
        ("python3 calc.py prints 5 5.0", ran.stdout.strip() == "5 5.0", ran.stdout.strip() or ran.stderr[-200:]),
        ("divide(1, 0) has a guard", probe not in ("ZeroDivisionError division by zero", ""), probe),
    ]
    display(brand.table("Check the agent's work", ["Check", "Result", "Detail"], [
        (label, brand.chip("Pass", "ok") if ok else brand.chip("Fail", "bad"), brand.mono(detail))
        for label, ok, detail in work]))
    assert all(ok for _, ok, _ in work), "The agent didn't fix calc.py. Rerun the cell."

### 7. Publish the Microsoft Learn tools

Claude Code can call MCP tools through the gateway too. The cell reuses the `learn` MCP server from lab 04, or deploys
it with lab 04's [tool-server.bicep](../04-federate-mcp-tools/infra/tool-server.bicep). Then it lists the tools the
way Claude Code does: `initialize`, then `tools/list`, with the key as a bearer token.

In [ ]:
created_server = lab.arm("GET", f"{lab.workspace_id}/toolServers/{server_name}") is None
if created_server:
    lab.deploy("../04-federate-mcp-tools/infra/tool-server.bicep", gatewayName=lab.gateway_name, serverName=server_name)
mcp_url = f"{lab.gateway_url}/default/toolservers/{server_name}/mcp"


def mcp_status():
    # Send an MCP initialize with the key, and return the HTTP status
    return requests.post(mcp_url, timeout=60, headers={
        "Authorization": f"Bearer {api_key}", "Content-Type": "application/json", "Accept": "application/json, text/event-stream"},
        json={"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {
            "protocolVersion": "2025-03-26", "capabilities": {}, "clientInfo": {"name": "aigateway-labs", "version": "1.0"}}},
    ).status_code


assert utils.wait_for_status(mcp_status, 200) == 200, "The MCP endpoint doesn't accept the key yet. Rerun the cell."
headers, _ = utils.mcp_session(mcp_url, api_key)
tools = utils.mcp_request(mcp_url, headers, "tools/list")["result"]["tools"]
display(brand.table(f"Tools on the {server_name} MCP server", ["Tool", "Description"], [
    (brand.mono(tool["name"]), " ".join(tool.get("description", "").split())[:120]) for tool in tools],
    sub=mcp_url, badge=brand.chip(f"{len(tools)} tools", "brand")))

### 8. Give Claude Code the Learn tools

`claude mcp add` adds the server to Claude Code's user settings, which live in the temporary `CLAUDE_CONFIG_DIR`. The
header `Authorization: Bearer ${AI_GATEWAY_API_KEY}` stays a reference in the file, and Claude Code reads the key from its
environment each time it connects. `--header` comes last, because it takes one or more values. `claude mcp list`
checks the connection.

Claude Code names the server's tools `mcp__ai-gateway-learn__<tool>`, and `claude -p` can't ask before it calls one,
so `--allowedTools "mcp__ai-gateway-learn"` allows them up front. The answer differs between runs.

In [ ]:
question = ("Use the Microsoft Learn tools to find the Azure Functions Core Tools command that creates a new Python "
            "function project. Answer in two sentences, with the command and the link to the page.")

if claude:
    subprocess.run([claude, "mcp", "remove", "ai-gateway-learn", "-s", "user"], env=claude_env(), capture_output=True)
    added = subprocess.run([claude, "mcp", "add", "--transport", "http", "--scope", "user", "ai-gateway-learn", mcp_url,
                            "--header", "Authorization: Bearer ${AI_GATEWAY_API_KEY}"],
                           env=claude_env(), capture_output=True, text=True)
    assert added.returncode == 0, added.stderr
    listed = subprocess.run([claude, "mcp", "list"], cwd=project, env=claude_env(), capture_output=True, text=True,
                            timeout=120).stdout
    line = next((line for line in listed.splitlines() if line.startswith("ai-gateway-learn")), "")
    connected = "Connected" in line
    display(brand.info("Claude Code's MCP servers", [("ai-gateway-learn", brand.mono(line.split(": ", 1)[-1]))],
                       sub="claude mcp list",
                       badge=brand.chip("Connected", "ok") if connected else brand.chip("Not connected", "bad")))
    assert connected, listed

    learn_run = run_claude(question, "--allowedTools", "mcp__ai-gateway-learn")
    show_run(learn_run, "Claude Code · with the Learn tools")
    assert any(tool.startswith("mcp__ai-gateway-learn__") for tool in learn_run["tools"]), \
        "Claude Code didn't call the Learn tools. Rerun the cell."
    cites = "learn.microsoft.com" in learn_run["answer"]
    display(brand.note("The answer cites Microsoft Learn" if cites else "The answer has no Learn link this time",
                       "ok" if cites else "warn"))

### 9. Lower the Haiku token limit

Coding agents can spend tokens fast. The gateway counts the tokens of each caller, here the `claude-code` key, so
the SDK and Claude Code share one counter. Lower the Haiku alias's token limit to 1,000 tokens per minute for each
caller to see what happens when it runs out. `set_policies` reads the alias and writes it back with new policies.
The change takes up to a minute to reach the gateway.

In [ ]:
def set_policies(tier, policies):
    # Write the alias back with new policies, and return them
    alias = lab.arm("GET", alias_path(tier))["properties"]
    body = {name: alias[name] for name in ("aliasName", "backendModels", "selectionOrder") if name in alias}
    return lab.arm("PUT", alias_path(tier), {"properties": {**body, "policies": policies}})["properties"]["policies"]


ALLOWANCE = 1000
low = [{**p, "count": ALLOWANCE} if p["type"] == "tokenLimit" else p for p in original["haiku"]]
display(brand.policies(aliases["haiku"], set_policies("haiku", low)))
brand.wait(60, "for the token limit to reach the gateway")

### 10. Burst Haiku until 429

Send requests that each use about 330 tokens. The first three or four succeed, then the gateway answers `429` with
`Retry-After`. The gateway counts a request's tokens after it answers, so the request that takes the total past the
allowance still succeeds. If no request is throttled, the policy is still applying, and the cell tries again.

In [ ]:
BURST_PROMPT = "Write a 150-word note to a platform team about why coding agents need token limits."


def burst(attempt, count=8):
    # Send requests back to back. A live table shows each status and the tokens used against the allowance
    board = brand.Board(f"Burst against {aliases['haiku']}", ["Request", "Status", "Retry-After", "Tokens used"],
                        sub=f"{key_name} · attempt {attempt}", budget=ALLOWANCE,
                        budget_label="Tokens used / allowance per minute")
    results = []
    for i in range(1, count + 1):
        r = send(aliases["haiku"], BURST_PROMPT, 300)
        board.add([i, brand.status(r["status"]), r["headers"].get("retry-after", "-"), r["tokens"] or "-"],
                  r["status"], r["tokens"])
        results.append(r)
        if sum(x["status"] == 429 for x in results) == 2:
            break
    return results


for attempt in range(1, 5):
    burst_results = burst(attempt)
    throttled = [r for r in burst_results if r["status"] == 429]
    if throttled:
        break
    display(brand.note("No 429 yet. The policy may still be applying.", "warn"))
    brand.wait(30, "before the next attempt")

assert throttled, "Expected at least one 429."

### 11. Chart the burst

Each bar is the tokens that one request used, and the line is the running total that the gateway compares with the
dashed allowance. Throttled requests never reach Claude, so they use no tokens and cost nothing.

In [ ]:
display(brand.usage_chart("Tokens per request against the allowance", [(r["status"], r["tokens"]) for r in burst_results],
                          limit=ALLOWANCE, limit_label=f"Allowance: {ALLOWANCE:,} tokens per minute",
                          sub=f"{aliases['haiku']} · {key_name}"))

### 12. Inspect a throttled response

The gateway answered, not Claude: the body names the token limit and how long to wait, and `Retry-After` gives the
same wait in seconds, until the caller's one-minute window frees up.

In [ ]:
t = throttled[-1]
display(brand.http(t["status"], t["headers"], t["body"], "A throttled response", sub=aliases["haiku"]))

### 13. See the 429 in Claude Code, then honor Retry-After

The cell waits for the window from the last step to free up, then asks Haiku for a long story. The gateway counts a
reply after it's sent, so the story arrives in full and leaves the key over the allowance for a minute or more: the
next request gets `429`. Claude Code shares the key's counter, so it gets the `429` too. It shows
`API Error: Request rejected (429) · Token limit is exceeded. Try again in N seconds.` and doesn't retry. The cell
waits the seconds that the gateway asked for, asks again, and gets an answer. Then it restores the Haiku limit to
100,000 tokens per minute.

In [ ]:
def retry_after(text, headers):
    # The seconds to wait: from the gateway's message, or from its Retry-After header
    found = re.search(r"Try again in (\d+) seconds", text or "")
    return int(found[1]) if found else int(headers.get("retry-after", 60))


STORY_PROMPT = "Write a 1,000-word short story about a gateway that guards a city."
PING = "Reply with the single word pong"

# Wait until the window from the last step frees up
r = send(aliases["haiku"], PING, 16)
while r["status"] == 429:
    brand.wait(retry_after(r["body"], r["headers"]) + 1, "for the window to free up")
    r = send(aliases["haiku"], PING, 16)

# One long story crosses the allowance: its reply still arrives, and the next request is throttled
story = send(aliases["haiku"], STORY_PROMPT, 1500)
r = send(aliases["haiku"], PING, 16)
rows = [("SDK · Haiku, a long story", story["status"], 200, f"{story['tokens'] or 0:,} tokens"),
        ("SDK · Haiku, the next request", r["status"], 429, json.loads(r["body"]).get("message", ""))]

if claude:
    run = run_claude(PING, "--model", "haiku")
    rows.append(("Claude Code · Haiku, allowance used up", run["status"], 429, run["answer"]))
    brand.wait(retry_after(run["answer"], r["headers"]) + 5, "as the gateway asked")
    run = run_claude(PING, "--model", "haiku")
    rows.append(("Claude Code · Haiku, after Retry-After", run["status"], 200, run["answer"]))
else:
    brand.wait(retry_after(None, r["headers"]) + 5, "as the gateway asked")
    rows.append(("SDK · Haiku, after Retry-After", send(aliases["haiku"], PING, 16)["status"], 200))

display(brand.checks("The token limit stops Claude Code until the window frees up", rows, sub=aliases["haiku"]))
display(brand.policies(aliases["haiku"], set_policies("haiku", original["haiku"])))
assert all(row[1] == row[2] for row in rows)

### 14. Give Opus a test budget

The gateway keeps one spend counter for each caller and each cost limit `id`. Keep the `id` when you change the
amount, and the spend so far still counts. Give the cost limit a new `id`, and the count starts again at $0. The cell
gives the Opus alias a test budget of $0.05 per day with a new `id`, so the burst in the next step starts from
nothing even though the earlier steps already spent on Opus. Step 16 puts the original budget back.

In [ ]:
TEST_BUDGET = 0.05
test_budget = {"type": "costLimit", "id": f"claude-code-test-{int(time.time())}", "amount": TEST_BUDGET,
               "period": "day", "counterKey": ["identity"]}
display(brand.policies(aliases["opus"], set_policies("opus", [p for p in original["opus"] if p["type"] != "costLimit"]
                                                    + [test_budget])))
brand.wait(60, "for the budget to reach the gateway")

### 15. Spend the Opus budget until 403

Send requests that each cost about a cent, until two are blocked. Each answer carries its estimated cost in
`x-alias-cost-per-day-consumed`, and the board adds them up against the budget. The call that takes the total past
the budget still succeeds, and the calls after it get `403`. Unlike the token limit, the budget doesn't free up after
a minute: the caller stays blocked until the next day starts, in UTC.

In [ ]:
SPEND_PROMPT = "Write a 300-word short story about a gateway that guards a city."


def spend(attempt, count=20):
    # Send requests until two are blocked. A live table shows each status and the spend against the budget
    board = brand.Board(f"Spend against {aliases['opus']}", ["Request", "Status", "Cost", "Remaining", "Retry after (s)"],
                        sub=f"{key_name} · attempt {attempt}", budget=TEST_BUDGET, unit="USD", stop=403,
                        budget_label="Spent / budget per day")
    results = []
    for i in range(1, count + 1):
        r = send(aliases["opus"], SPEND_PROMPT, 600)
        found = cost_headers(r["headers"])
        board.add([i, brand.status(r["status"]), brand.usd(r["cost"]), brand.usd(found.get("remaining")),
                   found.get("retry-after", "-")], r["status"], r["cost"])
        results.append(r)
        if sum(x["status"] == 403 for x in results) == 2:
            break
    return results


for attempt in range(1, 4):
    spend_results = spend(attempt)
    blocked = [r for r in spend_results if r["status"] == 403]
    if blocked:
        break
    display(brand.note("No 403 yet. The policy may still be applying.", "warn"))
    brand.wait(30, "before the next attempt")

assert blocked, "Expected at least one 403."
assert "cost quota" in blocked[-1]["body"].lower(), "A blocked response should say that the LLM cost quota is exceeded."
display(brand.usage_chart("Estimated cost per request against the budget",
                          [(r["status"], r["cost"]) for r in spend_results], limit=TEST_BUDGET,
                          limit_label=f"Budget: {brand.usd(TEST_BUDGET)} per day", unit="USD",
                          sub=f"{aliases['opus']} · {key_name}"))

### 16. Inspect a blocked response, and step down to Sonnet

A cost limit answers `403`, not `429`: retrying in a few seconds won't help. The body says
`LLM cost quota is exceeded` and when the budget renews, and `x-alias-cost-per-day-retry-after` gives the same wait in
seconds.

The budget belongs to the Opus alias, so the same key can still call Sonnet. `ask` tries Opus, and steps down to
Sonnet when the gateway answers `403`, so an app keeps working. Claude Code shows the `403` as
`Failed to authenticate. API Error: 403 LLM cost quota is exceeded` and suggests `/login`, which doesn't help:
switch to Sonnet with `--model sonnet`, or `/model sonnet` in a session. `--fallback-model` doesn't cover a `403`.
Then the cell puts the original budget back, with the `id` `claude-code-budget`, so today's earlier spend on Opus
counts again.

In [ ]:
b = blocked[-1]
display(brand.http(b["status"], b["headers"], b["body"], "A blocked response", sub=aliases["opus"]))


def ask(prompt, tiers=("opus", "sonnet")):
    # Ask each tier in turn, and step down to the next one when the gateway answers 403
    for tier in tiers:
        r = send(aliases[tier], prompt, 120)
        rows.append((f"SDK · {tier}", r["status"], 403 if tier == "opus" else 200,
                     r["text"] or json.loads(r["body"]).get("message", "")))
        if r["status"] != 403:
            return r


rows = []
answer = ask("In one sentence, why do teams give coding agents a budget?")
if claude:
    run = run_claude("Reply with the single word pong")
    rows.append(("Claude Code · default model, Opus", run["status"], 403, run["answer"]))
    run = run_claude("Reply with the single word pong", "--model", "sonnet")
    rows.append(("Claude Code · --model sonnet", run["status"], 200, run["answer"]))
display(brand.checks("The budget stops Opus, and Sonnet carries on", rows, sub=key_name))
display(brand.policies(aliases["opus"], set_policies("opus", original["opus"])))
assert all(row[1] == row[2] for row in rows)

### 17. Optional: sign in with Microsoft Entra ID instead of the key

If you have the **AI Gateway Models User** role on the gateway, as in
[Authorize callers with Microsoft Entra ID](../11-entra-rbac/README.md), callers can send a Microsoft Entra ID token
instead of the key. The SDK sends your token as the bearer token. Claude Code gets one from its `apiKeyHelper`
setting, which runs `az account get-access-token` and runs again every five minutes by default, so long sessions keep
working. The cell writes the setting to the temporary configuration, and leaves out `ANTHROPIC_AUTH_TOKEN`. Your
identity is another caller, so it gets its own token and cost counters. Without the role, the cell skips this step.

In [ ]:
oid, _, me = lab.signed_in_identity()
roles = [a["role"] for a in lab.role_assignments(oid)]
if {"AI Gateway Models User", "AI Gateway User"} & set(roles):
    entra = anthropic.Anthropic(base_url=base_url, auth_token=lab.token(utils.AIGW_SCOPE), max_retries=0)
    reply = send(aliases["sonnet"], "In one sentence, why is a short-lived Microsoft Entra ID token safer "
                 "than a long-lived API key?", 300, client=entra)
    assert reply["status"] == 200, reply["body"]
    display(brand.reply(reply["text"], "Messages API with your Microsoft Entra ID token", model=reply["model"],
                        tokens=reply["usage"], sub=me))
    if claude:
        helper = "az account get-access-token --scope https://ai.gateway.azure.com/Data.All --query accessToken -o tsv"
        settings_file = claude_home / "settings.json"
        settings_file.write_text(json.dumps({"apiKeyHelper": helper}, indent=2))
        try:
            run = run_claude("Reply with the single word pong", "--model", "sonnet",
                             settings={k: v for k, v in claude_settings.items() if k != "ANTHROPIC_AUTH_TOKEN"})
        finally:
            settings_file.unlink()
        show_run(run, "Claude Code · signed in with Microsoft Entra ID")
else:
    display(brand.note(f"{me} doesn't have AI Gateway Models User on the gateway: skipping. Lab 11 shows how to "
                       "assign it.", "neutral"))

### 18. Cut Claude Code off by suspending the key

Every developer who uses the `claude-code` key loses access with one change, on every machine, without touching their
settings. The cell suspends the key, waits until the model endpoint and the MCP endpoint both reject it with `401`,
then sets it back to `active` and checks that Claude Code works again. The key is reactivated even if a check fails.

The cell doesn't run Claude Code while the key is suspended: Claude Code retries a `401` for several minutes before it
gives up. If you set up monitoring in [lab 13](../13-monitoring/README.md), open **Monitoring** › **Models** in the
portal after a few minutes to see Claude Code's usage under the `claude-code` key.

In [ ]:
key_id = f"{lab.gateway_id}/apiKeys/{key_name}"


def set_key_state(state):
    return lab.arm("PATCH", key_id, {"properties": {"state": state}})["properties"]["state"]


def model_status():
    return send(aliases["haiku"], "Say hi in three words", 16)["status"]


rows = []
try:
    display(brand.note(f"API key {key_name} is {set_key_state('suspended')}", "warn"))
    rows.append(("Model endpoint, key suspended", utils.wait_for_status(model_status, 401), 401))
    rows.append(("MCP endpoint, key suspended", utils.wait_for_status(mcp_status, 401), 401))
finally:
    display(brand.note(f"API key {key_name} is {set_key_state('active')}", "ok"))

rows.append(("Model endpoint, key active", utils.wait_for_status(model_status, 200), 200))
rows.append(("MCP endpoint, key active", utils.wait_for_status(mcp_status, 200), 200))
if claude:
    run = run_claude("Say hi in three words")
    rows.append(("Claude Code, key active", run["status"], 200, run["answer"]))

display(brand.checks("One key switches Claude Code off and on", rows, sub=key_name))
assert all(row[1] == row[2] for row in rows)

### 19. Clean up resources

The cell deletes the temporary `CLAUDE_CONFIG_DIR`, and what this notebook created: the practice project, the
`claude-code` key, the `learn` MCP server, the aliases, the models, the provider, and the gateway's Foundry User
assignment. It puts back the policies of aliases that existed before, if a step left them changed. It keeps the
gateway, the Foundry account and its deployments, and anything that existed before the notebook ran. Your own Claude
Code settings are untouched: the notebook added nothing to them.

In [ ]:
shutil.rmtree(claude_home, ignore_errors=True)
if created_project:
    shutil.rmtree(project)
    display(brand.note(f"Deleted {project}", "ok"))

for tier in models:
    if tier not in created_aliases and alias_policies(tier) != original[tier]:
        set_policies(tier, original[tier])
        display(brand.note(f"Restored the policies on {aliases[tier]}", "ok"))

if created_key:
    lab.delete(f"{lab.gateway_id}/apiKeys/{key_name}")
if created_server:
    lab.delete(f"{lab.workspace_id}/toolServers/{server_name}")
for tier in created_aliases:
    lab.delete(alias_path(tier))
for tier in created_models:
    lab.delete(f"{lab.workspace_id}/modelProviders/{account}/models/{models[tier]}")
if created_provider:
    lab.delete(f"{lab.workspace_id}/modelProviders/{account}")
for assignment in created_role:
    lab.delete(assignment, api_version=utils.ROLES_API)

kept = [name for name, created in [(key_name, created_key), (server_name, created_server),
                                   (f"provider {account}", created_provider), (str(project), created_project)]
        if not created] + [aliases[tier] for tier in models if tier not in created_aliases]
if kept:
    display(brand.note(f"Kept what existed before this notebook ran: {', '.join(kept)}", "neutral"))

## Troubleshooting

- **`InvalidTemplateDeployment` with `RoleAssignmentExists`**: the gateway's managed identity already has Foundry User
  on the account under another assignment name. Step 2 checks for it, so rerun the cell. If the check can't see the
  assignment, pass `grantFoundryUser=False` to `lab.deploy`.
- **`HTTP 401` with the right key**: a new key can take a minute to work. Rerun the cell. If the key is still
  suspended after a failed run, rerun step 18, which sets it back to `active`.
- **`HTTP 404` with `model_not_found`**: the alias name doesn't match. Claude Code and the SDK send
  `<account>/<deployment>`, and the template publishes those names. New aliases take up to a minute to work.
- **The notebook skips the Claude Code steps**: `shutil.which("claude")` didn't find Claude Code. Install it, check
  that `claude --version` works in a terminal, then restart the editor so the kernel gets the new `PATH`.
- **Claude Code hangs**: an invalid or suspended key makes Claude Code retry the `401` for minutes. Check the key with
  the SDK cell in step 3, and that step 18 set it back to `active`.
- **No `429` or no `403` after several attempts**: policy changes take up to a minute to reach the gateway. Check the
  policies with `alias_policies("haiku")` or `alias_policies("opus")`, then rerun the cell.
- **Claude Code doesn't call the Learn tools**: check that step 7 lists tools, and that `claude mcp list` shows
  `✔ Connected`, then rerun the cell. The answer and the tools that Claude Code picks differ between runs.

For the other errors in this lab, see [Troubleshooting](../TROUBLESHOOTING.md#lab-25-claude-code).

## Next steps

- Run Claude Code interactively, connect it with the portal, and keep its settings in a file with the
  [lab guide](README.md).
- Give GitHub Copilot the same gateway in [lab 24](../24-github-copilot-byok/README.md) and its notebook
  [24-github-copilot-byok.ipynb](../24-github-copilot-byok/24-github-copilot-byok.ipynb).
- Tune token limits and budgets with overrides for some keys in [lab 06](../06-token-limits/README.md) and
  [lab 07](../07-cost-budgets/README.md).
- Choose which Learn tools Claude Code sees in [lab 05](../05-mcp-tool-controls/README.md) and its notebook
  [05-mcp-tool-controls.ipynb](../05-mcp-tool-controls/05-mcp-tool-controls.ipynb).
- Docs: [Claude Code LLM gateway configuration](https://code.claude.com/docs/en/llm-gateway) and
  [Claude Code settings](https://code.claude.com/docs/en/settings).
- Read this lab on the labs site: [https://azure.github.io/aigateway/labs/25-claude-code/](https://azure.github.io/aigateway/labs/25-claude-code/).